# 03 - Consistency Validator Fine-Tuning

**Purpose**: Fine-tune NLI model for Lumis consistency scoring

**Base Model**: `cross-encoder/nli-deberta-v3-xsmall`

**Transformation**:
- Original NLI: `(premise, hypothesis) → entailment/neutral/contradiction`
- Lumis Format: `(context+prompt, response) → same labels`
- Consistency Score: `1 - P(contradiction)`

In [ ]:
# Install dependencies
!pip install -q datasets accelerate sentencepiece scikit-learn openai tqdm nest_asyncio --upgrade typing_extensions>=4.12.0 torch torchvision torchaudio transformers pydantic pydantic-core


In [ ]:
import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## STEP 1: Prepare Training Data

Load and reformat NLI datasets:
- **SNLI**: Stanford Natural Language Inference (filter label != -1)
- **ANLI**: Adversarial Natural Language Inference

In [ ]:
# Load SNLI dataset
print("Loading SNLI dataset...")
snli = load_dataset("stanfordnlp/snli", trust_remote_code=True)

# Filter out invalid labels (-1)
def filter_valid_labels(example):
    return example['label'] != -1

snli_train = snli['train'].filter(filter_valid_labels).shuffle(seed=42).select(range(5000)) # OPTIMIZATION
snli_val = snli['validation'].filter(filter_valid_labels)
snli_test = snli['test'].filter(filter_valid_labels)

print(f"SNLI Train: {len(snli_train)} examples")
print(f"SNLI Val: {len(snli_val)} examples")
print(f"SNLI Test: {len(snli_test)} examples")

In [ ]:
# Load ANLI dataset
print("\nLoading ANLI dataset...")
anli = load_dataset("facebook/anli", trust_remote_code=True)

# ANLI has rounds r1, r2, r3
anli_train = concatenate_datasets([anli['train_r1'], anli['train_r2'], anli['train_r3']])
anli_val = concatenate_datasets([anli['dev_r1'], anli['dev_r2'], anli['dev_r3']])
anli_test = concatenate_datasets([anli['test_r1'], anli['test_r2'], anli['test_r3']])

# OPTIMIZATION: Subsample ANLI
anli_train = anli_train.shuffle(seed=42).select(range(5000))

print(f"ANLI Train: {len(anli_train)} examples")
print(f"ANLI Val: {len(anli_val)} examples")
print(f"ANLI Test: {len(anli_test)} examples")

## STEP 1.1: Generate Hard Negatives (Adversarial Data)

We generate 'Hard Negatives' to teach the model subtle semantic contradictions.
**Strategy**: Use DeepSeek V3 to modify SNLI/ANLI premises to create contradictions.

- **Mechanism**: `Diversity Engine` logic.
- **Concurrency**: `asyncio` with Semaphore(100).
- **Target**: 20% of training data.

In [ ]:
import asyncio
import os
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm
import nest_asyncio
from datasets import Dataset

# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

# Configuration
DEEPSEEK_API_KEY = "sk-fece6533d7a946f2967ff5899cab8538"
if not DEEPSEEK_API_KEY:
    print("WARNING: DEEPSEEK_API_KEY not found. Skipping hard negative generation (Demo Mode).")
    HARD_NEGATIVES_COUNT = 0
else:
    HARD_NEGATIVES_COUNT = 500  # Scale this up for real training
    print(f"Target Hard Negatives: {HARD_NEGATIVES_COUNT}")

async def generate_hard_negative(client, context, semaphore):
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": "You are a logic engine. Generate a SUBTLE contradiction for the statement. Change 1 entity or time or action. Output ONLY the contradiction."},
                    {"role": "user", "content": f"Statement: {context}"}
                ],
                temperature=0.7,
                max_tokens=64
            )
            contradiction = response.choices[0].message.content.strip()
            
            # Format as Lumis Input directly or keep generic?
            # We'll match the transform_to_lumis_format expectation later or manually construct here.
            # Let's return raw dictionary matching formatting logic
            return {
                'premise': context,
                'hypothesis': contradiction,
                'label': 2  # Contradiction
            }
        except Exception:
            return None

async def batch_generate_negatives(contexts):
    if not DEEPSEEK_API_KEY: return []
    
    client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
    semaphore = asyncio.Semaphore(100)  # High concurrency
    
    tasks = [generate_hard_negative(client, ctx, semaphore) for ctx in contexts]
    results = await tqdm.gather(*tasks, desc="Generating Hard Negatives")
    return [r for r in results if r is not None]

# EXECUTE GENERATION
hard_negatives_data = []
if DEEPSEEK_API_KEY:
    # Sample contexts from SNLI
    seed_contexts = [ex['premise'] for ex in snli_train.select(range(HARD_NEGATIVES_COUNT))]
    
    # Run async loop
    hard_negatives_data = await batch_generate_negatives(seed_contexts)
    print(f"Generated {len(hard_negatives_data)} hard negatives.")
else:
    print("Skipping generation.")

# Create HF Dataset from results
if hard_negatives_data:
    hard_negatives_dataset = Dataset.from_list(hard_negatives_data)
else:
    hard_negatives_dataset = None


## STEP 2: Transform to Lumis Format

Convert NLI format to Lumis consistency format:
```
[CONTEXT] {premise} [PROMPT] Based on the context, evaluate: [RESPONSE] {hypothesis}
```

In [ ]:
def transform_to_lumis_format(example):
    """
    Transform NLI example to Lumis consistency format.
    
    Original: premise, hypothesis → label
    Lumis: [CONTEXT] premise [PROMPT] query [RESPONSE] hypothesis → label
    
    Labels remain: 0=entailment, 1=neutral, 2=contradiction
    """
    premise = example['premise']
    hypothesis = example['hypothesis']
    label = example['label']
    
    # Construct Lumis-style input
    lumis_input = f"[CONTEXT] {premise} [PROMPT] Verify consistency of the following statement: [RESPONSE] {hypothesis}"
    
    return {
        'text': lumis_input,
        'premise': premise,
        'hypothesis': hypothesis,
        'label': label
    }

# Test transformation
sample = snli_train[0]
transformed = transform_to_lumis_format(sample)
print("Original:")
print(f"  Premise: {sample['premise']}")
print(f"  Hypothesis: {sample['hypothesis']}")
print(f"  Label: {sample['label']}")
print(f"\nTransformed:")
print(f"  {transformed['text'][:200]}...")

In [ ]:
# Apply transformation to all datasets
print("Transforming datasets to Lumis format...")

snli_train_lumis = snli_train.map(transform_to_lumis_format)
snli_val_lumis = snli_val.map(transform_to_lumis_format)
anli_train_lumis = anli_train.map(transform_to_lumis_format)
anli_val_lumis = anli_val.map(transform_to_lumis_format)

datasets_to_combine = [snli_train_lumis, anli_train_lumis]

# Process Hard Negatives
if 'hard_negatives_dataset' in locals() and hard_negatives_dataset is not None:
    print("Processing Hard Negatives...")
    hard_negatives_lumis = hard_negatives_dataset.map(transform_to_lumis_format)
    print("Aligning features to resolve mismatch...")
    hard_negatives_lumis = hard_negatives_lumis.cast(snli_train_lumis.features)
    datasets_to_combine.append(hard_negatives_lumis)

# Combine datasets
train_dataset = concatenate_datasets(datasets_to_combine)
val_dataset = concatenate_datasets([snli_val_lumis, anli_val_lumis])

print(f"\nCombined Train: {len(train_dataset)} examples")
print(f"Combined Val: {len(val_dataset)} examples")

# Shuffle training data
train_dataset = train_dataset.shuffle(seed=42)


## STEP 3: Load Base Model and Tokenizer

In [ ]:
MODEL_NAME = "cross-encoder/nli-deberta-v3-xsmall"

print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    problem_type="single_label_classification"
)

# Label mapping for NLI
LABEL_MAP = {
    0: 'entailment',
    1: 'neutral', 
    2: 'contradiction'
}

print(f"\nModel loaded successfully")
print(f"Number of parameters: {model.num_parameters():,}")

In [ ]:
def tokenize_function(examples):
    """Tokenize the Lumis-formatted text."""
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

print("Tokenizing datasets...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=['text', 'premise', 'hypothesis'])
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=['text', 'premise', 'hypothesis'])

train_tokenized.set_format('torch')
val_tokenized.set_format('torch')

print(f"Tokenization complete")

## STEP 4: Fine-Tuning Configuration

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./lumis_consistency_validator",
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=2000,
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=4,
    report_to="none"
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
def compute_metrics(eval_pred):
    """Compute evaluation metrics."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    
    # Per-class F1
    f1_per_class = f1_score(labels, predictions, average=None)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'f1_entailment': f1_per_class[0],
        'f1_neutral': f1_per_class[1],
        'f1_contradiction': f1_per_class[2]
    }

In [ ]:
# Initialize trainer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized")

## STEP 5: Fine-Tune Model

In [ ]:
# Fine-tune the model
print("Starting fine-tuning...")
print("="*50)

train_result = trainer.train()

print("\nTraining complete!")
print(f"Final training loss: {train_result.training_loss:.4f}")

In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...")
eval_results = trainer.evaluate()

print("\nValidation Results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

## STEP 6: Lumis Consistency Scoring Function

In [ ]:
class LumisConsistencyValidator:
    """
    Consistency validator for Lumis framework.
    
    Computes consistency_score = 1 - P(contradiction)
    """
    
    def __init__(self, model, tokenizer, device=None):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.eval()
        
        # Label indices
        self.ENTAILMENT = 0
        self.NEUTRAL = 1
        self.CONTRADICTION = 2
    
    def compute_consistency(self, context: str, prompt: str, response: str) -> dict:
        """
        Compute consistency score for a (context, prompt, response) triple.
        
        Args:
            context: The context/premise
            prompt: The user prompt
            response: The model response to evaluate
            
        Returns:
            dict with consistency_score and detailed probabilities
        """
        # Format input in Lumis style
        text = f"[CONTEXT] {context} [PROMPT] {prompt} [RESPONSE] {response}"
        
        # Tokenize
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=256
        ).to(self.device)
        
        # Get predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[0]
        
        # Extract probabilities
        p_entailment = probs[self.ENTAILMENT].item()
        p_neutral = probs[self.NEUTRAL].item()
        p_contradiction = probs[self.CONTRADICTION].item()
        
        # Consistency score = 1 - P(contradiction)
        consistency_score = 1 - p_contradiction
        
        return {
            'consistency_score': consistency_score,
            'p_entailment': p_entailment,
            'p_neutral': p_neutral,
            'p_contradiction': p_contradiction,
            'classification': ['entailment', 'neutral', 'contradiction'][probs.argmax().item()]
        }
    
    def batch_consistency(self, examples: list) -> list:
        """
        Compute consistency scores for a batch of examples.
        
        Args:
            examples: List of dicts with 'context', 'prompt', 'response' keys
            
        Returns:
            List of consistency results
        """
        texts = [
            f"[CONTEXT] {ex['context']} [PROMPT] {ex['prompt']} [RESPONSE] {ex['response']}"
            for ex in examples
        ]
        
        inputs = self.tokenizer(
            texts,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=256
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
        
        results = []
        for i in range(len(examples)):
            p_contradiction = probs[i, self.CONTRADICTION].item()
            results.append({
                'consistency_score': 1 - p_contradiction,
                'p_contradiction': p_contradiction
            })
        
        return results

print("LumisConsistencyValidator class defined")

In [ ]:
# Test the consistency validator
validator = LumisConsistencyValidator(model, tokenizer)

# Test examples
test_cases = [
    {
        'context': 'The sky is blue during a clear day.',
        'prompt': 'What color is the sky?',
        'response': 'The sky appears blue when there are no clouds.',
        'expected': 'high consistency (entailment)'
    },
    {
        'context': 'The sky is blue during a clear day.',
        'prompt': 'What color is the sky?',
        'response': 'The sky is always red.',
        'expected': 'low consistency (contradiction)'
    },
    {
        'context': 'Paris is the capital of France.',
        'prompt': 'Tell me about Paris.',
        'response': 'Paris has many restaurants.',
        'expected': 'medium consistency (neutral)'
    }
]

print("Testing Lumis Consistency Validator")
print("="*60)

for i, test in enumerate(test_cases):
    result = validator.compute_consistency(
        test['context'],
        test['prompt'],
        test['response']
    )
    
    print(f"\nTest {i+1}: {test['expected']}")
    print(f"  Context: {test['context']}")
    print(f"  Prompt: {test['prompt']}")
    print(f"  Response: {test['response']}")
    print(f"  Consistency Score: {result['consistency_score']:.4f}")
    print(f"  Classification: {result['classification']}")
    print(f"  P(entail/neutral/contra): {result['p_entailment']:.3f}/{result['p_neutral']:.3f}/{result['p_contradiction']:.3f}")

## STEP 7: Save Fine-Tuned Model

In [ ]:
# Save the fine-tuned model
OUTPUT_DIR = "./lumis_consistency_validator_final"

print(f"Saving model to {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save validator config
import json

config = {
    'base_model': MODEL_NAME,
    'task': 'consistency_validation',
    'labels': LABEL_MAP,
    'consistency_formula': '1 - P(contradiction)',
    'input_format': '[CONTEXT] {context} [PROMPT] {prompt} [RESPONSE] {response}',
    'training_data': ['SNLI', 'ANLI'],
    'max_length': 256
}

with open(f"{OUTPUT_DIR}/lumis_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print("Model and configuration saved successfully!")

## Summary

### Model Architecture
- **Base**: `cross-encoder/nli-deberta-v3-xsmall`
- **Task**: 3-class classification (entailment, neutral, contradiction)

### Lumis Integration
- **Input Format**: `[CONTEXT] ... [PROMPT] ... [RESPONSE] ...`
- **Consistency Score**: `1 - P(contradiction)`
  - Score close to 1.0 → Response is consistent with context
  - Score close to 0.0 → Response contradicts context

### Usage in Lumis Pipeline
```python
from lumis.validators import LumisConsistencyValidator

validator = LumisConsistencyValidator.load('./lumis_consistency_validator_final')
result = validator.compute_consistency(context, prompt, response)
if result['consistency_score'] < 0.5:
    # Flag potential contradiction
    pass
```

In [ ]:
print("\n" + "="*60)
print("FINE-TUNING COMPLETE")
print("="*60)
print(f"\nModel saved to: {OUTPUT_DIR}")
print("\nNext steps:")
print("1. Integrate LumisConsistencyValidator into Lumis pipeline")
print("2. Use consistency_score for response validation")
print("3. Set threshold (e.g., 0.5) to flag contradictions")